# ZEUS：加载预训练模型

第一步只确认官方权重能在 CPU 上加载并完成一次推理。这里使用随机输入，不涉及 OpenML 数据。

官方源码：[gmum/zeus](https://github.com/gmum/zeus)，提交 `3f75d280fe7edbe12fb1a80b4058b010af115e65`；权重于 2026-09-29 从其 README 中的 [Google Drive 链接](https://drive.google.com/file/d/1D7uikacymUnmmMxjUjBuCNIomqhBWS67/view?usp=sharing)下载。权重里有当前推理代码不用的 `cluster_embedding.weight`，加载时会显示这一项。

In [ ]:
from pathlib import Path
import sys

import torch

root = Path.cwd().parent  # notebook 位于 notebooks/，内核也从这里启动
sys.path.insert(0, str(root / 'vendor' / 'zeus'))

from zeus.configs import GMMConfig
from zeus.initialziation import model_initialization

In [ ]:
torch.manual_seed(42)
config = GMMConfig(device='cpu', dist_based_logit=True)
model = model_initialization(config)
checkpoint = torch.load(root / 'models' / 'zeus.pt', map_location='cpu', weights_only=False)
load_result = model.load_state_dict(checkpoint['model'], strict=False)
model.eval()

x = torch.randn(8, 1, config.dim)
with torch.inference_mode():
    output = model(x, k=2)

embeddings = output[:-config.num_gaussians, 0, :]
print(f'Python: {sys.version.split()[0]}, PyTorch: {torch.__version__}')
print(f'输入: {tuple(x.shape)}, 样本向量: {tuple(embeddings.shape)}')
print(f'权重轮次: {checkpoint["epoch"]}, 数值有限: {torch.isfinite(embeddings).all().item()}')
print(f'未匹配的权重: {load_result}')